# Week 3: Prompts as Engineering Artifacts

This notebook loads the versioned prompt files, evaluates them over the same 10-case test suite, and measures exact-match category accuracy plus semantic similarity of the rationale. Set `GEMINI_API_KEY` to run the prompts against Gemini.

Dependencies: `sentence-transformers` and `openai` (used as the client for Gemini's OpenAI-compatible endpoint).


In [ ]:
import os, json, pathlib, time, hashlib, urllib.request

def gemini_chat(messages, model='gemini-2.5-flash-lite', max_retries=3, **kw):
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model=model, messages=messages, **kw)
            return response.choices[0].message.content
        except Exception as exc:
            if '429' not in str(exc) or attempt == max_retries - 1:
                raise
            time.sleep(2 * (attempt + 1))

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')


In [ ]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())

def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)

print('metrics ready (exact-match + semantic)')


## Part 1: Versioned prompts and a test suite
The prompt versions are stored as repository files. When the repo is available locally, the notebook reads them from disk. In Colab, it fetches them from the assignment branch before merge and from `main` after merge.


In [ ]:
PROMPT_FILES = ['support_ticket_v1.txt', 'support_ticket_v2.txt']
RAW_BASES = [
    'https://raw.githubusercontent.com/mschemerii/cosc-650-applied-llm-systems/week3assignment/week-03/prompts',
    'https://raw.githubusercontent.com/mschemerii/cosc-650-applied-llm-systems/main/week-03/prompts',
]

def load_prompt_file(filename):
    local_candidates = [
        pathlib.Path('prompts') / filename,
        pathlib.Path('week-03') / 'prompts' / filename,
    ]
    for path in local_candidates:
        if path.exists():
            return path.read_text(encoding='utf-8'), str(path)

    last_error = None
    for base in RAW_BASES:
        url = f'{base}/{filename}'
        try:
            with urllib.request.urlopen(url, timeout=15) as response:
                return response.read().decode('utf-8'), url
        except Exception as exc:
            last_error = exc

    raise FileNotFoundError(f'Could not load {filename} locally or from GitHub: {last_error}')

PROMPT_V1, source_v1 = load_prompt_file(PROMPT_FILES[0])
PROMPT_V2, source_v2 = load_prompt_file(PROMPT_FILES[1])
print('v1 prompt source:', source_v1)
print('v2 prompt source:', source_v2)

tests = [
  {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
  {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
  {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
  {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
  {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
  {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
  {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
  {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
  {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
  {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
]
print('prompt versions:', 2, '| test cases:', len(tests))


In [ ]:
FIX = {
  'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
  'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
}

CACHE_PATH = pathlib.Path('.week3_response_cache.json')
CACHE = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}

def cache_key(version, prompt, ticket):
    payload = json.dumps({'version': version, 'prompt': prompt, 'ticket': ticket}, sort_keys=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def parse_json_response(txt):
    cleaned = (txt or '').strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        if cleaned.lower().startswith('json'):
            cleaned = cleaned[4:].lstrip()
    try:
        data = json.loads(cleaned)
        return data.get('category',''), data.get('rationale','')
    except Exception:
        return '', txt or ''

def run_case(version, prompt, t):
    if not LIVE:
        return FIX[version][t['id']]

    key = cache_key(version, prompt, t['ticket'])
    if key in CACHE:
        cached = CACHE[key]
        return cached['category'], cached['rationale']

    txt = gemini_chat([
        {'role':'system','content': prompt},
        {'role':'user','content': 'Ticket: ' + t['ticket']}
    ], temperature=0)
    category, rationale = parse_json_response(txt)
    CACHE[key] = {'category': category, 'rationale': rationale, 'raw': txt}
    CACHE_PATH.write_text(json.dumps(CACHE, indent=2), encoding='utf-8')
    return category, rationale

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat,'rationale':why})
    acc = sum(r['exact'] for r in rows) / len(rows)
    mean_sem = sum(r['sem'] for r in rows) / len(rows)
    return acc, mean_sem, rows

acc1, sem1, r1 = score('v1', PROMPT_V1)
acc2, sem2, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {acc1:.0%} | mean semantic {sem1:.3f}')
print(f'v2 exact-match {acc2:.0%} | mean semantic {sem2:.3f}')


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. If the live edit produces no regression, report that result and the measured change instead.


In [ ]:
changed = False
for t in tests:
    row1 = next(r for r in r1 if r['id'] == t['id'])
    row2 = next(r for r in r2 if r['id'] == t['id'])
    if row1['exact'] != row2['exact']:
        changed = True
        verdict = 'IMPROVED' if row2['exact'] > row1['exact'] else 'REGRESSED'
        print(f"#{t['id']} {verdict} | expected={t['cat']!r} | v1 exact={row1['exact']:.0f}, sem={row1['sem']:.3f} | v2 exact={row2['exact']:.0f}, sem={row2['sem']:.3f}")
if not changed:
    print('No exact-match category changes between v1 and v2; report the measured metric changes and note that no regression occurred.')


## Part 5: Submit
Run the suite with `GEMINI_API_KEY` set for live calls, save the executed notebook, record the measured v1/v2 results in `research-note.md`, and open a pull request containing the notebook, versioned prompt files, results summary, and linked research note.
